# 📘 Module 06 — PyTorch from Scratch
## Chapter 6.5 — A Complete Training Loop from Scratch

> **Learning Path:** Zero to Hero — AI Engineer  
> **Module:** 06 of 18  
> **Topics:** The 5-Step Canonical Training Heartbeat, Training vs. Validation Modes (`model.train()` vs `model.eval()`), Full MNIST Training & Metric Tracking, Model Checkpointing (`state_dict` Saving & Resuming)  
> **Format:** Notes first → Code second  

---


## 🎯 What You Will Learn

By the end of this chapter, you will be able to:

- [ ] Execute and explain the canonical 5-step PyTorch training heartbeat in exact order
- [ ] Articulate why `optimizer.zero_grad()` is mandatory and how gradient accumulation works under the hood
- [ ] Differentiate `model.train()` from `model.eval()` and understand how Dropout/BatchNorm behavior changes
- [ ] Leverage `torch.no_grad()` to halve validation memory consumption and eliminate graph overhead
- [ ] Build a robust, modular multi-epoch training and validation loop tracking epoch loss and accuracy
- [ ] Train a PyTorch neural network on real MNIST digits and generate convergence curves
- [ ] Save, bundle, and restore complete model checkpoints (`state_dict`, optimizer state, epoch metrics)

**Estimated Reading Time:** ~35 minutes


## 🗂️ Table of Contents

1. [The 5-Step Canonical Training Heartbeat](#1-the-5-step-canonical-training-heartbeat)
2. [Training vs. Validation Modes & Context Management](#2-training-vs-validation-modes--context-management)
3. [End-to-End Training on MNIST with Multi-Epoch Metric Tracking](#3-end-to-end-training-on-mnist-with-multi-epoch-metric-tracking)
4. [Model Checkpointing: State Dicts, Saving & Resuming Training](#4-model-checkpointing-state-dicts-saving--resuming-training)
5. [Chapter Summary](#5-chapter-summary)
6. [Resources & Further Reading](#6-resources--further-reading)
7. [Next Module Preview](#7-next-module-preview)


---

## 1. The 5-Step Canonical Training Heartbeat

---

### 🗣️ In Simple English

The PyTorch training loop is basically the heartbeat of modern deep learning — a repetitive 5-step cycle that turns raw guesses into superhuman predictions.

Think of it this way: It is like an archer shooting at a target with a coach watching. The archer shoots (forward pass), the coach measures how far off the bullseye the arrow landed (loss calculation), the archer clears their mind of previous muscle memory (zero gradients), the coach explains which muscles caused the error (backward pass / gradient calculation), and the archer adjusts their stance for the next shot (optimizer step).

The key point is: PyTorch does NOT automatically wipe out previous gradients or update weights — you must explicitly orchestrate the forward, loss, zero_grad, backward, and step operations in strict chronological sequence.

---

### 🌍 Real-Life Example

**Scenario:** Mastering a musical piece on the piano with a strict music instructor.

| Real Life | Tech Equivalent |
|-----------|----------------|
| Sheet music notes for a bar | Input mini-batch tensor ($X$) |
| Playing the chords on the keys | Model forward pass (`outputs = model(inputs)`) |
| How out-of-tune or discordant it sounded | Loss computation (`loss = criterion(outputs, targets)`) |
| Erasing yesterday's chalk notes on the chalkboard | Gradient buffer reset (`optimizer.zero_grad()`) |
| Instructor pointing out which fingers slipped | Automatic differentiation (`loss.backward()`) |
| Adjusting finger muscle placement for the next try | Parameter update (`optimizer.step()`) |

**Why this matters:** In every state-of-the-art AI system from GPT-4 to AlphaFold, the underlying weight optimization boils down to this exact 5-step heartbeat repeated millions of times.

---

### 📖 How It Works (Step by Step)

1. **Step 1: Forward Pass (`outputs = model(inputs)`):** The mini-batch of features flows layer-by-layer through weights and activations. PyTorch dynamically constructs a directed acyclic computation graph recording every mathematical transformation.
2. **Step 2: Loss Computation (`loss = criterion(outputs, targets)`):** A scalar penalty metric (such as Mean Squared Error or Cross-Entropy) compares the model's raw logits against the ground truth labels. This scalar is the root node of the backward pass.
3. **Step 3: Gradient Zeroing (`optimizer.zero_grad()`):** PyTorch's default design is to *accumulate* (add) newly computed gradients into each parameter's `.grad` tensor (`param.grad += new_grad`). Calling `zero_grad()` wipes these buffers clean so past mini-batches do not contaminate the current update.
4. **Step 4: Backward Pass (`loss.backward()`):** The autograd engine traverses the dynamic computation graph in reverse from the loss node, applying the chain rule of calculus to compute $\frac{\partial \mathcal{L}}{\partial w}$ for every parameter with `requires_grad=True`.
5. **Step 5: Optimizer Step (`optimizer.step()`):** The optimizer (SGD, Adam, RMSprop) updates each parameter tensor in-place using its calculated gradient according to its optimization algorithm rule (e.g., $w \leftarrow w - \eta \cdot \nabla w$).

---

### 🔑 Key Concepts

| Concept | What It Means | Why It Matters |
|---------|---------------|----------------|
| **Forward Pass** | Passing inputs through the model layers to produce predictions | Produces output logits and establishes the dynamic computation graph tape |
| **Loss Function** | Scalar mathematical function measuring discrepancy from ground truth | Serves as the single root starting scalar for backpropagation |
| **Gradient Accumulation** | PyTorch accumulates gradients (`+=`) rather than overwriting | Enables training giant effective batch sizes across limited GPU/CPU memory |
| **`optimizer.zero_grad()`** | Overwrites all parameter `.grad` attributes with zeros (or `None`) | Prevents gradients from previous batches bleeding into the current step |
| **`loss.backward()`** | Traverses computation graph in reverse to populate `.grad` tensors | Implements reverse-mode automatic differentiation via chain rule |
| **`optimizer.step()`** | Modifies parameter values in-place using their accumulated `.grad` | Executes the actual parameter update step of the learning algorithm |

---

### 🗺️ Visual Overview

```
                 =======================================
                 THE CANONICAL 5-STEP PYTORCH HEARTBEAT
                 =======================================

   [ Mini-Batch Data: (X, y) ]
               │
               ▼
   ┌────────────────────────────────┐
   │ 1. Forward Pass                │ ────► Builds Dynamic Autograd Graph
   │    outputs = model(X)          │
   └────────────────────────────────┘
               │
               ▼
   ┌────────────────────────────────┐
   │ 2. Compute Loss                │ ────► Scalar Root Node for Backprop
   │    loss = criterion(outputs, y)│
   └────────────────────────────────┘
               │
               ▼
   ┌────────────────────────────────┐
   │ 3. Zero Gradients              │ ────► Clears param.grad buffers
   │    optimizer.zero_grad()       │       (prevents batch bleeding)
   └────────────────────────────────┘
               │
               ▼
   ┌────────────────────────────────┐
   │ 4. Backward Pass               │ ────► Autograd Chain Rule Traversal
   │    loss.backward()             │       (populates param.grad)
   └────────────────────────────────┘
               │
               ▼
   ┌────────────────────────────────┐
   │ 5. Optimizer Step              │ ────► In-place Parameter Updates
   │    optimizer.step()            │       w = w - lr * grad
   └────────────────────────────────┘
               │
               ▼
     [ Repeat for Next Mini-Batch ]
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|--------|-----------|
| `optimizer.step()` automatically clears gradients after updating. | **False.** Gradients persist in `param.grad` until you explicitly call `optimizer.zero_grad()`. |
| `loss.backward()` updates model parameters directly. | **False.** `loss.backward()` ONLY computes derivatives and fills `.grad`. `optimizer.step()` actually changes the weight values. |
| You must call `zero_grad()` strictly at the beginning of the loop. | **False.** As long as `zero_grad()` occurs *before* `loss.backward()`, the gradient math is identical. Putting it before `backward()` is standard. |
| `optimizer.zero_grad(set_to_none=True)` is slower than default zeroing. | **False.** Setting `.grad = None` is faster and consumes less memory because it skips writing zeros to memory buffers. |

---

### ✅ When to Use This

- **Use when:** Training any neural network from scratch in PyTorch.
- **Use when:** Implementing custom training loops, adversarial training, or multi-task optimization.
- **Use when:** Simulating large batch sizes via gradient accumulation over multiple steps before calling `step()`.
- **Avoid when:** Performing validation or inference (gradients are completely unnecessary during evaluation).

---

### 📊 Section Quick Reference

| Term | Definition | Example |
|------|-----------|---------|
| `outputs = model(x)` | Runs input tensor through network layers | `logits = model(batch_x)` |
| `loss = criterion(pred, y)` | Evaluates prediction error | `loss = nn.CrossEntropyLoss()(logits, batch_y)` |
| `optimizer.zero_grad()` | Flushes gradient accumulation buffers | `optimizer.zero_grad(set_to_none=True)` |
| `loss.backward()` | Computes gradients via backpropagation | `loss.backward()` |
| `optimizer.step()` | Updates weights using computed gradients | `optimizer.step()` |


In [1]:
# ==============================================================================
# SECTION 1: THE 5-STEP CANONICAL TRAINING HEARTBEAT IN ISOLATION
# ==============================================================================
# Concept: Forward -> Loss -> Zero Grad -> Backward -> Optimizer Step
# Demonstrating: Parameter inspection before & after step, and gradient accumulation.
# ------------------------------------------------------------------------------

import torch
import torch.nn as nn
import torch.optim as optim

# Set manual seed for 100% reproducible demonstration
torch.manual_seed(42)

# Define a minimal linear model: y = x * W + b
# Input dimension: 3, Output dimension: 1
model = nn.Linear(in_features=3, out_features=1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

print("--- INITIAL MODEL PARAMETERS ---")
# Inspect the initial weight matrix and bias vector
for name, param in model.named_parameters():
    print(f"  {name}: value = {param.data.squeeze().tolist()}, grad = {param.grad}")

# Create a synthetic mini-batch of 2 samples
inputs = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=torch.float32)
targets = torch.tensor([[10.0], [20.0]], dtype=torch.float32)

print("\n--- EXECUTING CANONICAL 5-STEP HEARTBEAT ---")

# Step 1: Forward Pass (calculate predictions)
outputs = model(inputs)
print(f"1. Forward Pass Outputs:\n   {outputs.squeeze().detach().tolist()}")

# Step 2: Loss Computation (evaluate error)
loss = criterion(outputs, targets)
print(f"2. Computed Loss: {loss.item():.4f}")

# Step 3: Zero Gradients (flush previous buffers)
optimizer.zero_grad()
print(f"3. Gradients zeroed: weight.grad is {model.weight.grad}")

# Step 4: Backward Pass (calculate gradients via autograd)
loss.backward()
print(f"4. Backward Pass Complete! Calculated Gradients:")
print(f"   weight.grad = {model.weight.grad.tolist()}")
print(f"   bias.grad   = {model.bias.grad.tolist()}")

# Step 5: Optimizer Step (in-place parameter update: W = W - lr * grad)
optimizer.step()
print("\n5. Optimizer Step Executed! Updated Parameters:")
for name, param in model.named_parameters():
    print(f"   {name}: updated value = {param.data.squeeze().tolist()}")

print("\n--- DEMONSTRATING GRADIENT ACCUMULATION (OVOIDING ZERO_GRAD) ---")
# If we run backward again without zero_grad(), gradients SUM up!
outputs2 = model(inputs)
loss2 = criterion(outputs2, targets)
loss2.backward()  # Accumulates onto existing weight.grad!
print(f"   Accumulated weight.grad (2 passes without zero_grad):")
print(f"   {model.weight.grad.tolist()} (notice gradients doubled!)")

# Clean up gradients properly
optimizer.zero_grad()
print(f"   After optimizer.zero_grad(): weight.grad is {model.weight.grad}")



--- INITIAL MODEL PARAMETERS ---
  weight: value = [0.4414064884185791, 0.47920525074005127, -0.1352572739124298], grad = None
  bias: value = 0.5303604602813721, grad = None

--- EXECUTING CANONICAL 5-STEP HEARTBEAT ---
1. Forward Pass Outputs:
   [1.524405598640442, 3.8804688453674316]
2. Computed Loss: 165.8375
3. Gradients zeroed: weight.grad is None
4. Backward Pass Complete! Calculated Gradients:
   weight.grad = [[-72.95372009277344, -97.54884338378906, -122.14397430419922]]
   bias.grad   = [-24.59512710571289]

5. Optimizer Step Executed! Updated Parameters:
   weight: updated value = [7.736778259277344, 10.234089851379395, 12.079139709472656]
   bias: updated value = 2.989873170852661

--- DEMONSTRATING GRADIENT ACCUMULATION (OVOIDING ZERO_GRAD) ---
   Accumulated weight.grad (2 passes without zero_grad):
   [[534.8076171875, 705.22705078125, 875.6464233398438]] (notice gradients doubled!)
   After optimizer.zero_grad(): weight.grad is None


---

## 2. Training vs. Validation Modes & Context Management

---

### 🗣️ In Simple English

Mode switching is basically telling your neural network whether it is currently in "practice rehearsal" mode (`model.train()`) or "opening night performance" mode (`model.eval()`).

Think of it this way: During football practice, players wear heavy resistance vests and scrimmage with random simulated interference (Dropout/noise). During the championship match, they take off the weights and perform with 100% precision and consistency.

The key point is: `model.eval()` does NOT freeze model weights; it only alters the internal behavior of layers like Dropout and BatchNorm, while `torch.no_grad()` shuts down gradient tracking to save massive amounts of RAM.

---

### 🌍 Real-Life Example

**Scenario:** Taking a student driver car out with dual steering controls vs driving solo on your driver's license test.

| Real Life | Tech Equivalent |
|-----------|----------------|
| Driving instructor occasionally tapping brakes or perturbing wheel | `model.train()` (stochastic dropout randomly zeroing neurons) |
| Solo driving test where performance must be deterministic | `model.eval()` (dropout turned off, deterministic scaled weights) |
| Turning off the flight data recorder / black box to save storage | `torch.no_grad()` (autograd engine stops building graph tape) |
| Reviewing your total driver score at the end of the circuit | Validation epoch metric tracking (averaging loss across batches) |

**Why this matters:** Leaving a model in `train()` during testing leads to artificially degraded evaluation scores due to random neuron dropout, while forgetting `torch.no_grad()` causes Out-Of-Memory (OOM) crashes on validation splits.

---

### 📖 How It Works (Step by Step)

1. **Step 1: Switch to Evaluation Mode (`model.eval()`):** Recursively sets the internal `training` boolean flag on every layer to `False`. For `nn.Dropout`, this disables dropping and scales weights appropriately. For `nn.BatchNorm2d`, this stops computing batch statistics and uses frozen running mean and variance.
2. **Step 2: Enter No-Grad Context (`with torch.no_grad():`):** Tells PyTorch's C++ autograd engine to temporarily suspend graph tape construction. Intermediate activations are not saved for backward passes, halving memory consumption and speeding up computation.
3. **Step 3: Execute Validation Forward Pass:** Inputs pass through the network deterministically. Predictions are obtained without any gradient tracking overhead.
4. **Step 4: Accumulate Validation Metrics:** Batch losses and correct classifications are summed across all validation batches.
5. **Step 5: Restore Training Mode (`model.train()`):** Before resuming the next training epoch, `model.train()` must be invoked to reactivate dropout and batch norm tracking.

---

### 🔑 Key Concepts

| Concept | What It Means | Why It Matters |
|---------|---------------|----------------|
| **`model.train()`** | Sets model and all submodules into training mode | Activates dropout randomization and batch norm running statistics updates |
| **`model.eval()`** | Sets model and all submodules into evaluation mode | Deactivates dropout and freezes batch norm running statistics |
| **`torch.no_grad()`** | Context manager disabling autograd graph tracking | Halves GPU/CPU memory consumption and boosts inference speed |
| **`torch.inference_mode()`** | A stricter, even faster alternative to `no_grad()` | Disables view tracking and version counters for maximum inference speed |
| **Generalization Gap** | Gap between training accuracy and validation accuracy | Diagnostic indicator of overfitting (large gap) vs underfitting |

---

### 🗺️ Visual Overview

```
TRAINING MODE (model.train()):
┌────────────────┐     ┌────────────────┐     ┌────────────────┐
│ Input Tensor   │ ──► │ Dropout Layer  │ ──► │ Dynamic Graph  │ ──► Gradients computed
│ (Mini-batch)   │     │ (Drops 50% p)  │     │ Tape RECORDING │     RAM used for backward
└────────────────┘     └────────────────┘     └────────────────┘

EVALUATION MODE (model.eval() + torch.no_grad()):
┌────────────────┐     ┌────────────────┐     ┌────────────────┐
│ Input Tensor   │ ──► │ Dropout Layer  │ ──► │ Dynamic Graph  │ ──► NO Gradients
│ (Validation)   │     │ (100% Pass-thru│     │ Tape DISABLED  │     50% LESS RAM!
│                │     │  Deterministic)│     │ (No recording) │     Deterministic outputs
└────────────────┘     └────────────────┘     └────────────────┘
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|--------|-----------|
| `model.eval()` freezes model parameters so they cannot be updated. | **False.** `model.eval()` does NOT freeze parameters; it only toggles layer behavior (Dropout/BatchNorm). Calling `loss.backward()` and `optimizer.step()` still updates weights unless `torch.no_grad()` is used or params are frozen. |
| `torch.no_grad()` automatically sets `model.eval()`. | **False.** They are completely independent. `torch.no_grad()` disables gradient calculation; `model.eval()` alters layer behavior. You must use BOTH during validation. |
| You can calculate validation accuracy inside the training loop without `eval()`. | **False.** If you do, Dropout will randomly zero out predictions, artificially depressing your validation metrics. |

---

### ✅ When to Use This

- **Use `model.train()` when:** Running mini-batch optimization and computing gradients.
- **Use `model.eval()` + `torch.no_grad()` when:** Running validation loops, test set evaluations, and production inference.
- **Use `torch.inference_mode()` when:** Deploying production serving pipelines in PyTorch 2.0+ for maximum throughput.

---

### 📊 Section Quick Reference

| Command | Effect on Layers | Effect on Autograd | Use Case |
|---------|------------------|-------------------|----------|
| `model.train()` | Dropout active, BatchNorm tracks stats | Graph tracking active | Training epochs |
| `model.eval()` | Dropout inactive, BatchNorm frozen | Graph tracking active unless wrapped | Validation / Testing |
| `with torch.no_grad():` | Layer behavior unchanged | Disables graph recording & gradients | Validation passes |
| `with torch.inference_mode():` | Layer behavior unchanged | Disables autograd + version tracking | Production serving |


In [2]:
# ==============================================================================
# SECTION 2: TRAINING VS. EVALUATION MODES & CONTEXT MANAGEMENT
# ==============================================================================
# Concept: model.train() vs model.eval() with Dropout, and torch.no_grad()
# Demonstrating: Output stochasticity vs determinism, and autograd tape disabling.
# ------------------------------------------------------------------------------

import torch
import torch.nn as nn

# Set seed for reproducible dropout sampling
torch.manual_seed(42)

# Build a small network with Dropout (p = 0.5)
class DropoutDemoNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 4 inputs -> 4 outputs with 50% probability of dropping neurons
        self.fc = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)
        # Initialize with identity-like weights for clarity
        nn.init.eye_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        x = self.fc(x)
        x = self.dropout(x)
        return x

demo_net = DropoutDemoNet()
sample_input = torch.tensor([[10.0, 20.0, 30.0, 40.0]], dtype=torch.float32)

print("--- 1. TRAINING MODE: model.train() ---")
demo_net.train()
print(f"Network is in training mode: {demo_net.training}")

# Forward pass #1 in train mode
out_train_1 = demo_net(sample_input)
# Forward pass #2 with EXACT same input
out_train_2 = demo_net(sample_input)

print(f"Pass 1 output: {out_train_1.tolist()[0]}")
print(f"Pass 2 output: {out_train_2.tolist()[0]}")
print("-> Notice: Different neurons were dropped to 0, remaining scaled by 1/(1-0.5) = 2.0!")

print("\n--- 2. EVALUATION MODE: model.eval() ---")
demo_net.eval()
print(f"Network is in training mode: {demo_net.training}")

# Forward pass #1 in eval mode
out_eval_1 = demo_net(sample_input)
# Forward pass #2 in eval mode
out_eval_2 = demo_net(sample_input)

print(f"Pass 1 output: {out_eval_1.tolist()[0]}")
print(f"Pass 2 output: {out_eval_2.tolist()[0]}")
are_identical = torch.equal(out_eval_1, out_eval_2)
print(f"Are eval outputs 100% deterministic and identical? {are_identical}")

print("\n--- 3. AUTOGRAD CONTEXT: WITH VS WITHOUT torch.no_grad() ---")
# Normal forward pass (records computation graph)
out_with_grad = demo_net(sample_input)
print(f"Without no_grad() context:")
print(f"  requires_grad: {out_with_grad.requires_grad}")
print(f"  grad_fn:       {out_with_grad.grad_fn} (autograd graph recorded!)")

# Forward pass inside torch.no_grad() (skips computation graph)
with torch.no_grad():
    out_no_grad = demo_net(sample_input)
    print(f"Inside with torch.no_grad():")
    print(f"  requires_grad: {out_no_grad.requires_grad}")
    print(f"  grad_fn:       {out_no_grad.grad_fn} (tape completely disabled - RAM saved!)")



--- 1. TRAINING MODE: model.train() ---
Network is in training mode: True
Pass 1 output: [20.0, 40.0, 0.0, 0.0]
Pass 2 output: [20.0, 0.0, 60.0, 0.0]
-> Notice: Different neurons were dropped to 0, remaining scaled by 1/(1-0.5) = 2.0!

--- 2. EVALUATION MODE: model.eval() ---
Network is in training mode: False
Pass 1 output: [10.0, 20.0, 30.0, 40.0]
Pass 2 output: [10.0, 20.0, 30.0, 40.0]
Are eval outputs 100% deterministic and identical? True

--- 3. AUTOGRAD CONTEXT: WITH VS WITHOUT torch.no_grad() ---
Without no_grad() context:
  requires_grad: True
  grad_fn:       <AddmmBackward0 object at 0x000001987C2AD840> (autograd graph recorded!)
Inside with torch.no_grad():
  requires_grad: False
  grad_fn:       None (tape completely disabled - RAM saved!)


---

## 3. End-to-End Training on MNIST with Multi-Epoch Metric Tracking

---

### 🗣️ In Simple English

An end-to-end training loop is basically an automated assembly line that trains a neural network over multiple complete passes (epochs) of a dataset, continuously testing its progress on unseen validation data.

Think of it this way: It is like an athlete going through pre-season training camp. Each camp day (epoch) consists of morning drills with the playbook (training batches) and an evening scrimmage against another team (validation set) to confirm they are genuinely improving rather than just memorizing practice drills.

The key point is: We monitor both training and validation metrics across epochs to verify the network is truly generalizing to new data rather than simply memorizing training examples (overfitting).

---

### 🌍 Real-Life Example

**Scenario:** A student studying for university finals using textbook problem sets (train) and practice mock exams (validation).

| Real Life | Tech Equivalent |
|-----------|----------------|
| Textbook problem sets with solutions | Training dataset batches |
| Working through problems and correcting mistakes | Training loop forward-backward updates |
| Taking a timed practice exam without peeking | Validation loop (`model.eval()` + `torch.no_grad()`) |
| Recording test scores on a wall chart each week | Metric tracking across epochs (loss & accuracy curves) |
| The official final exam at semester end | Final test set evaluation |

**Why this matters:** This loop architecture is the exact template used across the AI industry for training computer vision models, language models, and recommendation engines.

---

### 📖 How It Works (Step by Step)

1. **Step 1: Dataset & DataLoader Setup:** Load training and validation datasets (here, real MNIST digits loaded strictly from our local `D:` drive data directory) wrapped in PyTorch `DataLoader` objects for batching and shuffling.
2. **Step 2: Model Architecture Definition:** Build a Multi-Layer Perceptron (MLP) with `nn.Linear`, `nn.ReLU`, and `nn.Dropout` layers.
3. **Step 3: Loss & Optimizer Configuration:** Select `nn.CrossEntropyLoss` (which combines Softmax + Negative Log-Likelihood with maximum numerical stability) and an adaptive optimizer like `torch.optim.Adam`.
4. **Step 4: Training Step per Batch:** For each batch in `train_loader`, execute the 5-step heartbeat, accumulating running loss and correct prediction counts.
5. **Step 5: Validation Step per Epoch:** After the training epoch finishes, switch to `model.eval()`, disable gradients with `torch.no_grad()`, iterate through `val_loader`, and compute unbiased validation loss and accuracy.
6. **Step 6: Metric Logging & Visualization:** Store epoch loss and accuracy in history lists and plot dual convergence curves.

---

### 🔑 Key Concepts

| Concept | What It Means | Why It Matters |
|---------|---------------|----------------|
| **Epoch** | One complete traversal through the entire training dataset | Measures macro-level learning progression over time |
| **Batch Loss vs Epoch Loss** | Loss on a single slice of data vs average loss across all samples | Epoch loss smooths out mini-batch variance to show true learning trajectory |
| **`nn.CrossEntropyLoss`** | Combines LogSoftmax and NLLLoss in a single numerically stable class | Expects raw logits; mathematically avoids exponent overflow/underflow |
| **`torch.argmax(dim=1)`** | Finds the index of the highest logit per sample | Converts model probability scores into predicted class indices (0-9) |
| **Learning Curves** | Plots of train vs validation loss and accuracy over epochs | Essential diagnostic tool to detect overfitting, underfitting, or divergence |

---

### 🗺️ Visual Overview

```
                      MODULAR TRAINING LOOP ARCHITECTURE
                      ==================================

  ┌───────────────────────────────────────────────────────────┐
  │ For Epoch in range(1, num_epochs + 1):                    │
  │                                                           │
  │   1. TRAIN PHASE:                                         │
  │      model.train()                                        │
  │      For batch (X, y) in train_loader:                    │
  │         outputs = model(X)                                │
  │         loss = criterion(outputs, y)                      │
  │         optimizer.zero_grad()                             │
  │         loss.backward()                                   │
  │         optimizer.step()                                  │
  │         track train_loss, train_correct                   │
  │                                                           │
  │   2. VALIDATION PHASE:                                    │
  │      model.eval()                                         │
  │      with torch.no_grad():                                │
  │         For batch (X_val, y_val) in val_loader:           │
  │            val_outputs = model(X_val)                     │
  │            val_loss = criterion(val_outputs, y_val)       │
  │            track val_loss, val_correct                    │
  │                                                           │
  │   3. LOG & REPORT:                                        │
  │      Print: Epoch E | Train Loss / Acc | Val Loss / Acc   │
  │      Append to history for plotting loss curves           │
  └───────────────────────────────────────────────────────────┘
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|--------|-----------|
| Calculating accuracy as `(preds == targets).float().mean()` per batch and averaging batch accuracies is exact. | **False.** If the last batch is smaller than standard batch size, a simple average of batch accuracies produces a mathematically skewed average. True epoch accuracy must sum total correct samples divided by total dataset length. |
| You must apply Softmax activation to the final layer before `nn.CrossEntropyLoss`. | **False.** PyTorch's `nn.CrossEntropyLoss` expects *raw logits*. Applying Softmax manually will double-apply the function and degrade training stability. |
| A training loss near 0 always means your model is performing wonderfully. | **False.** If training loss is 0.01 but validation loss is 1.50, your model has severely overfitted and memorized noise. |

---

### ✅ When to Use This

- **Use when:** Training any supervised deep learning classifier or regressor.
- **Use when:** Benchmarking architectures against baseline datasets (like MNIST, CIFAR, or tabular datasets).
- **Use when:** Establishing a clean, debuggable training pipeline before adding distributed training or hyperparameter search.

---

### 📊 Section Quick Reference

| Operation | PyTorch Code | Explanation |
|-----------|--------------|-------------|
| Model Prediction Class | `preds = torch.argmax(logits, dim=1)` | Class index with highest score |
| Correct Count | `(preds == targets).sum().item()` | Integer count of accurate predictions |
| Running Loss Accumulation | `running_loss += loss.item() * batch_size` | Weighted total loss accumulator |
| True Epoch Metric | `epoch_loss = running_loss / total_samples` | Exact unskewed dataset metric |


In [3]:
# ==============================================================================
# SECTION 3: END-TO-END TRAINING ON REAL MNIST WITH METRIC TRACKING
# ==============================================================================
# Concept: Modular train/val loops, learning curves, and visual test predictions.
# Demonstrating: Training an MLP on MNIST directly from D: drive data storage.
# ------------------------------------------------------------------------------

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Set seeds for 100% reproducible training
torch.manual_seed(42)
np.random.seed(42)

# Ensure output assets directory exists on D: drive
os.makedirs("Module-06-PyTorch-Foundations/assets", exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Dataset & DataLoader Setup (Loading MNIST from local D: drive)
# ------------------------------------------------------------------------------
class LocalMNISTDataset(Dataset):
    """Custom PyTorch Dataset wrapping local mnist.npz on D: drive."""
    def __init__(self, data_path, split="train", max_samples=None):
        raw = np.load(data_path)
        images = raw["x_train"] if split == "train" else raw["x_test"]
        labels = raw["y_train"] if split == "train" else raw["y_test"]

        # Subsample if requested for fast, crisp notebook execution
        if max_samples is not None:
            images = images[:max_samples]
            labels = labels[:max_samples]

        # Normalize pixel values from [0, 255] to [0.0, 1.0] and flatten to 784
        self.images = torch.tensor(images, dtype=torch.float32).view(-1, 28 * 28) / 255.0
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

mnist_path = os.path.join("Module-06-PyTorch-Foundations", "data", "mnist.npz")
print(f"Loading local MNIST from: {mnist_path}")

# Use 6,000 train samples and 1,000 validation samples for fast CPU convergence
train_dataset = LocalMNISTDataset(mnist_path, split="train", max_samples=6000)
val_dataset = LocalMNISTDataset(mnist_path, split="test", max_samples=1000)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Loaded {len(train_dataset)} training samples ({len(train_loader)} batches)")
print(f"Loaded {len(val_dataset)} validation samples ({len(val_loader)} batches)")

# ------------------------------------------------------------------------------
# 2. Model Architecture: Multi-Layer Perceptron (MLP)
# ------------------------------------------------------------------------------
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 784 pixels -> Hidden 1: 128 -> Hidden 2: 64 -> Output: 10 classes
        self.network = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),       # Regularization during training
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)        # 10 output class logits
        )

    def forward(self, x):
        return self.network(x)

model = MNISTClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

# ------------------------------------------------------------------------------
# 3. Training & Validation Execution Loop
# ------------------------------------------------------------------------------
num_epochs = 5
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print("\n" + "=" * 65)
print(f"{'Epoch':^7} | {'Train Loss':^12} | {'Train Acc':^11} | {'Val Loss':^12} | {'Val Acc':^11}")
print("=" * 65)

for epoch in range(1, num_epochs + 1):
    # --- A. TRAINING PHASE ---
    model.train()
    running_train_loss = 0.0
    running_train_correct = 0

    for batch_x, batch_y in train_loader:
        # Step 1: Forward pass
        logits = model(batch_x)
        # Step 2: Loss computation
        loss = criterion(logits, batch_y)
        # Step 3: Zero gradients
        optimizer.zero_grad(set_to_none=True)
        # Step 4: Backward pass
        loss.backward()
        # Step 5: Optimizer step
        optimizer.step()

        # Track batch statistics
        running_train_loss += loss.item() * len(batch_y)
        predictions = torch.argmax(logits, dim=1)
        running_train_correct += (predictions == batch_y).sum().item()

    epoch_train_loss = running_train_loss / len(train_dataset)
    epoch_train_acc = (running_train_correct / len(train_dataset)) * 100.0

    # --- B. VALIDATION PHASE ---
    model.eval()
    running_val_loss = 0.0
    running_val_correct = 0

    with torch.no_grad():
        for val_x, val_y in val_loader:
            val_logits = model(val_x)
            v_loss = criterion(val_logits, val_y)

            running_val_loss += v_loss.item() * len(val_y)
            val_preds = torch.argmax(val_logits, dim=1)
            running_val_correct += (val_preds == val_y).sum().item()

    epoch_val_loss = running_val_loss / len(val_dataset)
    epoch_val_acc = (running_val_correct / len(val_dataset)) * 100.0

    # Record history
    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)

    print(f" {epoch:^5} | {epoch_train_loss:^12.4f} | {epoch_train_acc:^10.2f}% | {epoch_val_loss:^12.4f} | {epoch_val_acc:^10.2f}%")

print("=" * 65)

# ------------------------------------------------------------------------------
# 4. Plotting & Saving Training Curves
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, num_epochs + 1)

# Loss Curve
axes[0].plot(epochs_range, history["train_loss"], marker="o", color="#1f77b4", label="Training Loss", linewidth=2)
axes[0].plot(epochs_range, history["val_loss"], marker="s", color="#ff7f0e", label="Validation Loss", linewidth=2)
axes[0].set_title("Cross-Entropy Loss vs. Epochs", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Loss", fontsize=11)
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend(fontsize=10)

# Accuracy Curve
axes[1].plot(epochs_range, history["train_acc"], marker="o", color="#2ca02c", label="Training Accuracy", linewidth=2)
axes[1].plot(epochs_range, history["val_acc"], marker="s", color="#d62728", label="Validation Accuracy", linewidth=2)
axes[1].set_title("Classification Accuracy (%) vs. Epochs", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Accuracy (%)", fontsize=11)
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend(fontsize=10)

plt.tight_layout()
curve_asset_path = os.path.join("Module-06-PyTorch-Foundations", "assets", "training_validation_curves.png")
plt.savefig(curve_asset_path, dpi=120)
plt.close()
print(f"\nSaved training curves to: {curve_asset_path}")

# ------------------------------------------------------------------------------
# 5. Visualizing Test Predictions on Unseen MNIST Digits
# ------------------------------------------------------------------------------
model.eval()
with torch.no_grad():
    sample_images, sample_labels = val_dataset.images[:10], val_dataset.labels[:10]
    sample_logits = model(sample_images)
    sample_preds = torch.argmax(sample_logits, dim=1)
    probabilities = torch.softmax(sample_logits, dim=1)

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
axes = axes.flatten()

for i in range(10):
    img = sample_images[i].view(28, 28).numpy()
    true_lbl = sample_labels[i].item()
    pred_lbl = sample_preds[i].item()
    conf = probabilities[i, pred_lbl].item() * 100.0

    axes[i].imshow(img, cmap="gray")
    is_correct = (true_lbl == pred_lbl)
    title_color = "#2ca02c" if is_correct else "#d62728"
    axes[i].set_title(f"Pred: {pred_lbl} ({conf:.1f}%)\nTrue: {true_lbl}", color=title_color, fontsize=10, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("MNIST Test Sample Predictions (Green = Correct, Red = Wrong)", fontsize=14, fontweight="bold")
plt.tight_layout()
pred_asset_path = os.path.join("Module-06-PyTorch-Foundations", "assets", "mnist_test_predictions.png")
plt.savefig(pred_asset_path, dpi=120)
plt.close()
print(f"Saved test predictions grid to: {pred_asset_path}")



Loading local MNIST from: Module-06-PyTorch-Foundations\data\mnist.npz
Loaded 6000 training samples (94 batches)
Loaded 1000 validation samples (16 batches)

 Epoch  |  Train Loss  |  Train Acc  |   Val Loss   |   Val Acc  
   1   |    0.8245    |   75.88   % |    0.4142    |   88.80   %
   2   |    0.3307    |   90.17   % |    0.3446    |   89.60   %
   3   |    0.2433    |   92.62   % |    0.2835    |   91.30   %
   4   |    0.1913    |   94.25   % |    0.2500    |   92.10   %
   5   |    0.1553    |   95.23   % |    0.2468    |   92.10   %

Saved training curves to: Module-06-PyTorch-Foundations\assets\training_validation_curves.png
Saved test predictions grid to: Module-06-PyTorch-Foundations\assets\mnist_test_predictions.png


---

## 4. Model Checkpointing: State Dicts, Saving & Resuming Training

---

### 🗣️ In Simple English

Model checkpointing is basically hitting "Save Game" during a multi-hour video game campaign so you can turn off the computer, come back tomorrow, and resume playing exactly where you left off without losing your progress.

Think of it this way: If your computer crashes midway through baking a 10-layer cake, you do not want to throw away the 5 layers you already baked. A checkpoint preserves all the baked layers (weights), the timer and temperature settings (optimizer state), and which layer you were currently baking (epoch number).

The key point is: In PyTorch, always save the `state_dict` (a plain Python dictionary of learned weight tensors) rather than serializing the entire Python class object with pickle.

---

### 🌍 Real-Life Example

**Scenario:** Saving progress on a 500-page academic doctoral thesis document.

| Real Life | Tech Equivalent |
|-----------|----------------|
| Writing new paragraphs and refining arguments | Backpropagation weight updates (`optimizer.step()`) |
| Hitting `Ctrl + S` to save progress to cloud storage | `torch.save(checkpoint, 'checkpoint.pt')` |
| Preserved text sentences and revision history | Model `state_dict` and optimizer momentum buffers |
| Opening the saved document on another laptop | `model.load_state_dict(torch.load('checkpoint.pt'))` |
| Resuming writing from Chapter 4 where you stopped | Resuming training from saved `epoch + 1` |

**Why this matters:** Production model training jobs can run for days or weeks across GPU clusters. Without robust checkpointing, cloud preemptions, hardware faults, or power cuts would destroy hundreds of thousands of dollars of compute.

---

### 📖 How It Works (Step by Step)

1. **Step 1: Extract Model & Optimizer State Dictionaries:** Every PyTorch module and optimizer implements `.state_dict()`, which maps parameter names to their exact in-memory `torch.Tensor` data.
2. **Step 2: Assemble Complete Checkpoint Bundle:** Build a Python dictionary packing together:
   - `epoch`: The current completed epoch number.
   - `model_state_dict`: The learned weights and biases.
   - `optimizer_state_dict`: The optimizer's internal buffers (e.g., Adam's first and second momentum moments).
   - `best_val_loss`: The best validation score achieved so far.
3. **Step 3: Serialize to Disk (`torch.save`):** Save the dictionary to disk as a `.pt` or `.pth` file using `torch.save(checkpoint, path)` directly within your project directory on `D:`.
4. **Step 4: Restore Model State (`model.load_state_dict`):** Instantiate the blank model architecture, load the saved dictionary via `checkpoint = torch.load(path)`, and call `model.load_state_dict(checkpoint['model_state_dict'])`.
5. **Step 5: Resume Training:** Restore `optimizer.load_state_dict(checkpoint['optimizer_state_dict'])` to preserve momentum, and continue the training loop seamlessly.

---

### 🔑 Key Concepts

| Concept | What It Means | Why It Matters |
|---------|---------------|----------------|
| **`state_dict`** | Python dictionary mapping each layer name to its parameter tensor | The standard, framework-agnostic serialization format in PyTorch |
| **`torch.save()`** | Serializes a Python dictionary of tensors to disk via zip/pickle | Creates `.pt` or `.pth` checkpoint files |
| **`torch.load()`** | Deserializes saved file into memory on specified device | Safely reconstructs saved tensor state |
| **Complete Checkpoint** | Dictionary storing model weights, optimizer moments, epoch, and metrics | Enables seamless resumption of interrupted training runs |
| **`load_state_dict(strict=True)`** | Enforces exact name and shape match between saved weights and model | Catches architecture mismatches immediately |

---

### 🗺️ Visual Overview

```
                      PYTORCH CHECKPOINT WORKFLOW
                      ===========================

   TRAINED MODEL                         SAVED ON DISK (checkpoint.pt)
 ┌────────────────┐                      ┌────────────────────────────┐
 │ model.weight   │ ──state_dict()──►    │ 'model_state_dict': {      │
 │ model.bias     │                      │    'network.0.weight': ... │
 └────────────────┘                      │    'network.0.bias': ...   │
                                         │ },                         │
   OPTIMIZER STATE                       │ 'optimizer_state_dict':... │
 ┌────────────────┐                      │ 'epoch': 5,                │
 │ Adam moments   │ ──state_dict()──►    │ 'best_val_loss': 0.1824    │
 └────────────────┘                      └────────────────────────────┘
                                                       │
                                                 torch.load()
                                                       │
                                                       ▼
   FRESH / NEW MODEL                     RESTORED ACTIVE MODEL
 ┌────────────────┐                      ┌────────────────────────────┐
 │ Blank weights  │ ◄──load_state_dict()─│ Exact identical weights!   │
 │ (Random init)  │                      │ Predictions match to 10^-7 │
 └────────────────┘                      └────────────────────────────┘
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|--------|-----------|
| `torch.save(model, 'model.pt')` is better because it saves the architecture too. | **False.** Serializing the full model class relies on Python `pickle`, which easily breaks if the code is refactored, moved to another file, or loaded on another machine. Saving `model.state_dict()` is the official PyTorch best practice. |
| Restoring `model.state_dict()` is sufficient to resume training. | **False.** If using an optimizer with momentum (like Adam), failing to restore `optimizer.state_dict()` destroys learned momentum buffers, causing a temporary loss spike when resuming. |
| Checkpoint files must be saved on `C:` drive system paths. | **False.** Checkpoints can and should be saved in dedicated project subdirectories (e.g., `Module-06-PyTorch-Foundations/checkpoints/`) on your secondary storage drive (`D:`). |

---

### ✅ When to Use This

- **Use when:** Implementing Early Stopping (saving only when validation loss improves).
- **Use when:** Training large models across multiple hours or days to protect against crashes.
- **Use when:** Exporting trained weights for production deployment, model serving, or mobile conversion (ONNX, TorchScript).

---

### 📊 Section Quick Reference

| Task | PyTorch Code |
|------|--------------|
| Save Model Weights Only | `torch.save(model.state_dict(), 'weights.pt')` |
| Load Model Weights | `model.load_state_dict(torch.load('weights.pt'))` |
| Save Full Checkpoint | `torch.save({'epoch': e, 'model': model.state_dict(), 'opt': opt.state_dict()}, 'ckpt.pt')` |
| Restore Full Checkpoint | `ckpt = torch.load('ckpt.pt'); model.load_state_dict(ckpt['model']); opt.load_state_dict(ckpt['opt'])` |


In [4]:
# ==============================================================================
# SECTION 4: MODEL CHECKPOINTING: SAVING, VERIFYING, AND RESUMING
# ==============================================================================
# Concept: state_dict bundling, weight serialization, and exact state restoration.
# Demonstrating: Complete checkpoint round-trip and validation parity check.
# ------------------------------------------------------------------------------

import os
import torch
import torch.nn as nn
import torch.optim as optim

# Ensure checkpoints directory exists on D: drive
checkpoint_dir = os.path.join("Module-06-PyTorch-Foundations", "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_file = os.path.join(checkpoint_dir, "best_mnist_model.pt")

print("--- 1. INSPECTING STATE DICT OF TRAINED MODEL ---")
print(f"Model state_dict keys ({len(model.state_dict())} parameter tensors):")
for key, tensor in model.state_dict().items():
    print(f"  Key: {key:<25} | Shape: {str(list(tensor.shape)):<12} | Dtype: {tensor.dtype}")

# ------------------------------------------------------------------------------
# 2. Bundling & Saving the Full Training Checkpoint
# ------------------------------------------------------------------------------
checkpoint_payload = {
    "epoch": num_epochs,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_val_loss": history["val_loss"][-1],
    "best_val_acc": history["val_acc"][-1]
}

print(f"\nSaving complete checkpoint to: {checkpoint_file}")
torch.save(checkpoint_payload, checkpoint_file)
file_size_kb = os.path.getsize(checkpoint_file) / 1024.0
print(f"Checkpoint successfully saved! File size: {file_size_kb:.2f} KB")

# ------------------------------------------------------------------------------
# 3. Instantiating a Fresh, Uninitialized Model
# ------------------------------------------------------------------------------
fresh_untrained_model = MNISTClassifier()
fresh_untrained_model.eval()

# Compare predictions of trained model vs untrained model on test sample
test_input = val_dataset.images[:5]
with torch.no_grad():
    trained_preds = torch.argmax(model(test_input), dim=1)
    untrained_preds = torch.argmax(fresh_untrained_model(test_input), dim=1)

print("\n--- 2. VERIFYING UNTRAINED MODEL VS TRAINED MODEL ---")
print(f"Ground Truth Labels:   {val_dataset.labels[:5].tolist()}")
print(f"Trained Model Preds:   {trained_preds.tolist()} (high accuracy)")
print(f"Untrained Model Preds: {untrained_preds.tolist()} (random guessing)")

# ------------------------------------------------------------------------------
# 4. Loading Checkpoint and Restoring Weights
# ------------------------------------------------------------------------------
print(f"\n--- 3. RESTORING CHECKPOINT INTO FRESH MODEL ---")
loaded_checkpoint = torch.load(checkpoint_file)
fresh_untrained_model.load_state_dict(loaded_checkpoint["model_state_dict"])
fresh_untrained_model.eval()

with torch.no_grad():
    restored_preds = torch.argmax(fresh_untrained_model(test_input), dim=1)
    restored_logits = fresh_untrained_model(test_input)
    original_logits = model(test_input)

print(f"Restored Model Preds:  {restored_preds.tolist()}")
max_discrepancy = torch.max(torch.abs(restored_logits - original_logits)).item()
print(f"Max absolute logit difference between original and restored: {max_discrepancy:.2e}")
print(f"Exact mathematical parity achieved? {max_discrepancy < 1e-6}")

# ------------------------------------------------------------------------------
# 5. Restoring Optimizer State and Resuming Training
# ------------------------------------------------------------------------------
resumed_optimizer = optim.Adam(fresh_untrained_model.parameters(), lr=0.002)
resumed_optimizer.load_state_dict(loaded_checkpoint["optimizer_state_dict"])
start_epoch = loaded_checkpoint["epoch"] + 1

print(f"\n--- 4. RESUMING TRAINING FROM CHECKPOINT ---")
print(f"Successfully restored checkpoint from epoch {loaded_checkpoint['epoch']}.")
print(f"Ready to resume training starting at Epoch {start_epoch} with preserved optimizer moments!")



--- 1. INSPECTING STATE DICT OF TRAINED MODEL ---
Model state_dict keys (6 parameter tensors):
  Key: network.0.weight          | Shape: [128, 784]   | Dtype: torch.float32
  Key: network.0.bias            | Shape: [128]        | Dtype: torch.float32
  Key: network.3.weight          | Shape: [64, 128]    | Dtype: torch.float32
  Key: network.3.bias            | Shape: [64]         | Dtype: torch.float32
  Key: network.5.weight          | Shape: [10, 64]     | Dtype: torch.float32
  Key: network.5.bias            | Shape: [10]         | Dtype: torch.float32

Saving complete checkpoint to: Module-06-PyTorch-Foundations\checkpoints\best_mnist_model.pt
Checkpoint successfully saved! File size: 1289.79 KB

--- 2. VERIFYING UNTRAINED MODEL VS TRAINED MODEL ---
Ground Truth Labels:   [7, 2, 1, 0, 4]
Trained Model Preds:   [7, 2, 1, 0, 4] (high accuracy)
Untrained Model Preds: [0, 0, 0, 0, 0] (random guessing)

--- 3. RESTORING CHECKPOINT INTO FRESH MODEL ---
Restored Model Preds:  [7, 2, 1, 0

---

## 5. Chapter Summary

| Topic | Simple Definition | Key Takeaway |
|-------|------------------|--------------|
| **5-Step Canonical Heartbeat** | The exact 5-step loop: forward → loss → zero_grad → backward → step | PyTorch requires manual execution of each step; `zero_grad()` prevents batch gradient bleeding. |
| **`model.train()` vs `model.eval()`** | Toggling training mode vs evaluation mode on network submodules | Changes behavior of stochastic layers like Dropout and BatchNorm; does NOT freeze parameters. |
| **`torch.no_grad()`** | Context manager disabling the autograd computation graph tape | Halves memory consumption and accelerates inference during validation passes. |
| **Epoch Metric Tracking** | Tracking true total loss and accuracy across entire train and validation splits | True epoch accuracy requires summing total correct samples, not unweighted batch averages. |
| **Model Checkpointing** | Saving `model.state_dict()` and `optimizer.state_dict()` to disk | Serializing state dicts is portable, modular, and allows exact training resumption. |


## 🔗 Resources & Further Reading

| Resource | Type | Why Read It |
|----------|------|-------------|
| [PyTorch Official Tutorial: A 60 Minute Blitz](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html) | Official Guide | The foundational canonical walkthrough of PyTorch's core design philosophy |
| [PyTorch Documentation: Saving and Loading Models](https://pytorch.org/tutorials/beginner/saving_loading_models.html) | Documentation | Comprehensive best practices on state dictionaries and multi-GPU checkpoints |
| [Understanding PyTorch Autograd Engine](https://pytorch.org/docs/stable/autograd.html) | Technical Reference | Deep dive into graph lifecycle, backward hooks, and memory optimization |
| [Karpathy's Micrograd to PyTorch Walkthrough](https://www.youtube.com/watch?v=VMj-3S1tku0) | Video Lecture | Brilliant intuitive explanation of computational graphs and gradient backpropagation |


## ➡️ Next Module: Module 07 — Regularization & Training

Now that you have built a complete, production-grade PyTorch training loop from scratch, you are ready to tackle the techniques that prevent overfitting and supercharge training speed!

In **Module 07**, you will master:
- **Dropout & DropConnect:** Preventing co-adaptation of neurons
- **Weight Decay ($L_1$ and $L_2$ Regularization):** Penalizing excessive weight magnitudes
- **Weight Initialization Techniques:** Xavier/Glorot and He/Kaiming initializations
- **Batch Normalization & Layer Normalization:** Stabilizing internal covariate shift
- **Learning Rate Schedules:** StepLR, ExponentialLR, and Cosine Annealing with Warmup
- **Data Augmentation Strategies:** Geometric and photometric image transformations
